# SOXX 负半方差份额风险信息筛查

本 notebook 为唯一冻结计划的执行入口。仅使用前一完整 RTH 的一分钟 RV、负半方差份额和有符号收益，预测下一完整 RTH RV；B 控制 RV1、RV5、RV22 与有符号收益，C 仅增加 q。2018 仅提供回看，2019–2022 拟合两次固定 OLS，训练残差估计 Duan smearing。

执行须匹配 root GO、设计与实现评审、代码/依赖/测试和规范化 notebook 身份。每一阶段数值查询前封存精确来源、版本与有界 XNYS 日历；每次只查询 SOXX open/close/price_valid。完整分钟涵盖官方 390 或 210 分钟，不能跨缺失或隔夜拼接。

两个模型和截至 2022-12-30 的精确 22 交易日尾部保存至原生 Qlib Recorder，关闭并读回后才读取 2023+ 数值。任何 held 目标或所需 22 日滞后缺失，立即 needs_data，不执行 QLIKE 或 bootstrap。唯一主门为 q 系数>0、配对 QLIKE 改善均值>0、固定 63 日/10000 次/seed20261014/alpha .025/15 下界>0。全部 held 年度均报告。

这是复用历史的风险信息筛查；不证明费用后 SOXX 超额、独立验证或允许修改生产配置。


In [1]:
from pathlib import Path
import sys
ROOT = Path('/Users/rziiiii/dev/workspace')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.workflow.signed_variance_information import run
SUMMARY = run(root=ROOT)
SUMMARY


{
  "experiment": "signed_variance_information_20261020",
  "decision": "not_promoted",
  "recorder_id": "cd099f4ba3254914acfae4b0f5c30ff8",
  "training": {
    "target_sessions": 1008,
    "qualified_sessions": 985,
    "excluded_sessions": 23,
    "measured_sessions": 1062,
    "unknown_source_sessions": 1
  },
  "held": {
    "target_sessions": 946,
    "qualified_sessions": 946,
    "unknown_target_sessions": 0,
    "unknown_lag_windows": 0
  },
  "new_information_fits": 2,
  "new_bootstraps": 1,
  "new_native_accounts": 0,
  "information_screen": {
    "gamma": -0.7523944913196178,
    "held_sessions": 946,
    "mean_paired_qlike_improvement": 0.0002514446220943663,
    "stationary_bootstrap_lower": -0.0022351442662873295,
    "yearly": {
      "2023": {
        "sessions": 250,
        "mean_qlike_B": -7.585029641447931,
        "mean_qlike_C": -7.586234902329456,
        "mean_paired_improvement": 0.0012052608815244063
      },
      "2024": {
        "sessions": 252,
        "m

# SOXX 负半方差份额风险信息筛查

结果：**not_promoted，固定版本闭合**。Qlib Recorder：`cd099f4ba3254914acfae4b0f5c30ff8`；冻结回执：`6f38c2891ef01a37c4906c1199fce4b23f85bb0ea1bb9f55ed606c1ba6993083`。实际执行 2 次 OLS 拟合、1 次固定 bootstrap、0 个组合账户。

该筛查检验前一完整 RTH 的负半方差份额 q，是否在控制同一会话 RV1、RV5、RV22 和有符号收益后，增加下一完整 RTH RV 的预测信息。依据 [Patton 与 Sheppard（2015）](https://public.econ.duke.edu/~ap172/Patton_Sheppard_REStat_2015.pdf) 提出固定正系数假设；文献不证明本 SOXX 结果或费用后投资收益。原有五分钟 H5 配置路线和无符号 RV 持有区间路线保持闭合。

原生 Qlib 训练目标为 2019–2022 年：1008 个日历目标中 985 个合格、23 个按 B/C 共同掩码排除；训练来源有 1 个不合格会话。2018 仅提供回看。标准化与 Duan smearing 均仅使用训练数据；两模型与截至 2022-12-30 的精确 22 会话尾部关闭并读回后，才读取 held 数值。2023-01-03 至 2026-10-09 的 946/946 个目标及所需滞后全部合格，无 held 缺失日期压缩。

三个预登记主门中，两门失败：

- C 的 q 系数 gamma = -0.7523944913196178，未通过 gamma>0 的固定正号门。
- held 配对 QLIKE 改善均值 = 0.0002514446220944，通过均值>0 门。
- 固定 stationary bootstrap 下界 = -0.0022351442662873，未通过下界>0 门。参数为平均区块 63 会话、10000 次、seed 20261014、单侧分位数 .025/15。

| 年度 | 会话数 | 平均 QLIKE B | 平均 QLIKE C | 平均配对改善 B−C |
| --- | ---: | ---: | ---: | ---: |
| 2023 | 250 | -7.585029641448 | -7.586234902329 | 0.001205260882 |
| 2024 | 252 | -7.426517393462 | -7.427762807822 | 0.001245414360 |
| 2025 | 250 | -7.376550320459 | -7.376741753363 | 0.000191432904 |
| 2026（截至 10-09，部分年度） | 194 | -6.824063820228 | -6.821872318926 | -0.002191501302 |

2023、2024、2025 的描述性均值为正，2026 部分年度为负。所有年度均保留，未根据区间结果改选符号、状态、窗口、频率或目标。该固定机制的正号与统计支持不足，版本在最小可审计证据归档后停止；不把负系数改号或通过奖励/状态分组重新调参。

分钟来源为回溯 raw-NONE 快照，官方日历与完整分钟覆盖不认证历史首次接收。2023–2026 是复用研究历史，不构成独立验证。QLIKE 是风险预测损失；本卡未测试投资组合、IBKR 费用、滑点、相对 SOXX 净财富、年度/滚动收益稳定性或前向证据，因此不作经济收益或生产晋升声明。

完整原生 Dataset、会话/滞后掩码、模型、smearing、配对预测损失、bootstrap 与来源/日历身份保存在 Recorder；卡片保留 [summary.json](summary.json)、[verification_manifest.json](verification_manifest.json) 和 NAS 绑定。下一步由持续研究负责人另行冻结具备新机制的独立任务；本固定版本不再搜索或重复拟合。
